# Hệ thống dự đoán Customer Churn bằng Machine Learning

Notebook này được viết lại theo hướng cá nhân hóa cho đồ án của Nguyễn Trọng Nam. Phạm vi bám đúng đề cương: IBM Telco Customer Churn, tiền xử lý, EDA, Logistic Regression, Decision Tree, Random Forest, đánh giá bằng Accuracy/Precision/Recall/F1-score và chương trình dự đoán đơn giản.

Notebook **không sao chép** notebook/repo tham khảo; phần huấn luyện được đóng gói lại bằng `Pipeline` và `ColumnTransformer` để tái sử dụng cho CLI và Streamlit app.

## 1. Cài đặt và import thư viện

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.append(str(PROJECT_ROOT))

import pandas as pd
from sklearn.model_selection import train_test_split

from src.config import RAW_DATA_PATH, RANDOM_STATE, TEST_SIZE
from src.data_processing import clean_telco_data, load_telco_data, split_features_target
from src.modeling import train_and_evaluate_models
from src.visualization import (
    plot_target_distribution,
    plot_churn_by_category,
    plot_numeric_by_churn,
    plot_model_metrics,
    plot_confusion_matrix,
)


## 2. Tải và kiểm tra dữ liệu

In [2]:
raw_df = load_telco_data(RAW_DATA_PATH)
df = clean_telco_data(raw_df)

print('Kích thước dữ liệu:', df.shape)
display(df.head())
display(df.info())


Kích thước dữ liệu: (7043, 21)


,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,...,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,7590-VHVEG,Female,0,Yes,No,1,No,No phone service,DSL,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,29.85,29.85,No
1,5575-GNVDE,Male,0,No,No,34,Yes,No,DSL,Yes,...,Yes,No,No,No,One year,No,Mailed check,56.95,1889.50,No
2,3668-QPYBK,Male,0,No,No,2,Yes,No,DSL,Yes,...,No,No,No,No,Month-to-month,Yes,Mailed check,53.85,108.15,Yes
3,7795-CFOCW,Male,0,No,No,45,No,No phone service,DSL,Yes,...,Yes,Yes,No,No,One year,No,Bank transfer (automatic),42.30,1840.75,No
4,9237-HQITU,Female,0,No,No,2,Yes,No,Fiber optic,No,...,No,No,No,No,Month-to-month,Yes,Electronic check,70.70,151.65,Yes


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7043 entries, 0 to 7042
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   customerID        7043 non-null   object 
 1   gender            7043 non-null   object 
 2   SeniorCitizen     7043 non-null   int64  
 3   Partner           7043 non-null   object 
 4   Dependents        7043 non-null   object 
 5   tenure            7043 non-null   int64  
 6   PhoneService      7043 non-null   object 
 7   MultipleLines     7043 non-null   object 
 8   InternetService   7043 non-null   object 
 9   OnlineSecurity    7043 non-null   object 
 10  OnlineBackup      7043 non-null   object 
 11  DeviceProtection  7043 non-null   object 
 12  TechSupport       7043 non-null   object 
 13  StreamingTV       7043 non-null   object 
 14  StreamingMovies   7043 non-null   object 
 15  Contract          7043 non-null   object 
 16  PaperlessBilling  7043 non-null   object 


None

In [3]:
missing = df.isna().sum().rename('missing_count').to_frame()
missing['missing_rate'] = missing['missing_count'] / len(df)
display(missing.sort_values('missing_count', ascending=False).head(10))


,missing_count,missing_rate
TotalCharges,11,0.001562
gender,0,0.000000
SeniorCitizen,0,0.000000
Partner,0,0.000000
customerID,0,0.000000
Dependents,0,0.000000
tenure,0,0.000000
MultipleLines,0,0.000000
PhoneService,0,0.000000
OnlineSecurity,0,0.000000


## 3. Phân tích khám phá dữ liệu EDA

Các biểu đồ được lưu trong `reports/figures/` để dùng cho báo cáo.

In [4]:
plot_target_distribution(df)
for col in ['Contract', 'InternetService', 'PaymentMethod', 'TechSupport']:
    plot_churn_by_category(df, col)
for col in ['tenure', 'MonthlyCharges', 'TotalCharges']:
    plot_numeric_by_churn(df, col)

print('Đã tạo biểu đồ EDA trong reports/figures/')


Đã tạo biểu đồ EDA trong reports/figures/


## 4. Chuẩn bị dữ liệu cho mô hình

Biến mục tiêu được mã hóa: `Churn = Yes` thành 1 và `Churn = No` thành 0. Chia dữ liệu 80/20 và giữ tỷ lệ Churn bằng `stratify=y`.

In [5]:
X, y = split_features_target(df)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=RANDOM_STATE, stratify=y
)

print('Train:', X_train.shape, 'Test:', X_test.shape)
print('Tỷ lệ Churn trong train:', y_train.mean().round(4))
print('Tỷ lệ Churn trong test:', y_test.mean().round(4))


Train: (5634, 19) Test: (1409, 19)
Tỷ lệ Churn trong train: 0.2654
Tỷ lệ Churn trong test: 0.2654


## 5. Huấn luyện ba mô hình theo đề cương

In [6]:
trained_models, metrics_df = train_and_evaluate_models(X_train, X_test, y_train, y_test)
display(metrics_df)
plot_model_metrics(metrics_df)


,model,accuracy,precision,recall,f1_score
0,Random Forest,0.763662,0.538318,0.770053,0.633663
1,Decision Tree,0.745209,0.513043,0.788770,0.621707
2,Logistic Regression,0.738112,0.504303,0.783422,0.613613


WindowsPath('C:/Users/ASUS/Downloads/nam_telco_churn_project/reports/figures/model_metric_comparison.png')

## 6. Ma trận nhầm lẫn

In [7]:
for model_name, model in trained_models.items():
    plot_confusion_matrix(model, X_test, y_test, model_name)
print('Đã lưu confusion matrix cho từng mô hình.')


Đã lưu confusion matrix cho từng mô hình.


## 7. Lưu mô hình tốt nhất

In [8]:
import joblib

best_model_name = metrics_df.iloc[0]['model']
best_model = trained_models[best_model_name]
model_path = PROJECT_ROOT / 'models' / 'best_churn_model.joblib'
model_path.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(best_model, model_path)
metrics_df.to_csv(PROJECT_ROOT / 'reports' / 'model_comparison.csv', index=False)

print('Mô hình tốt nhất:', best_model_name)
print('Đường dẫn mô hình:', model_path)


Mô hình tốt nhất: Random Forest
Đường dẫn mô hình: c:\Users\ASUS\Downloads\nam_telco_churn_project\models\best_churn_model.joblib


## 8. Dự đoán thử một khách hàng mới

In [9]:
from src.data_processing import make_single_customer_example
from src.modeling import predict_churn_probability

customer = make_single_customer_example()
result = predict_churn_probability(best_model, customer)
display(pd.DataFrame([customer]))
display(result)


,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,Female,0,Yes,No,5,No,No phone service,DSL,No,Yes,No,No,No,No,Month-to-month,Yes,Electronic check,35.4,170.5


,churn_probability,prediction,prediction_label
0,0.652818,1,Yes


## 9. Chạy hệ thống dự đoán đơn giản

Sau khi huấn luyện, có thể chạy:

```bash
python predict.py
streamlit run app.py
```

`predict.py` phục vụ kiểm thử nhanh bằng dòng lệnh. `app.py` là giao diện nhập thông tin khách hàng và hiển thị xác suất Churn.